In [1]:
from IPython.display import display, HTML
display(HTML('<style>.container { width:80% !important; }</style>'))

In [2]:
import pandas as pd
import numpy as np
import os
import re
from datetime import datetime
import json

In [3]:
pd.set_option('display.max_columns', 500) # To utilise larger part of screen

pd.set_option('display.max_colwidth', None) # To show full cell text

In [4]:
data_path = 'data/pdl_test'

# Data Import

**Target structure**

People: Role data is for target organisation
- uuid
- Name
- (Profile picture)
- Title
- Seniority
- Started date
- Ended date

Companies: Both target organisations and subsequent ones, ex. PayPal but also companies supported by its alumni network
- uuid
- Name
- (Logo)
- Short description
- HQ country
- HQ city
- Foundation data
- Total funding (USD)
- IPO date
- IPO valuation (USD)
- Acquisition date
- Acquisition valuation (USD)
- Acquirer name

Experiences/relationships: Relationships between alumni from the target organisation and subsequent organisations
- Subsequent organisation uuid
- Person uuid
- Relationship type (ex. executive, investor, board member, advisor)
- Subsequent organisation title
- Subsequent organisation seniority
- Subsequent organisation started date
- Subsequent organisation ended date

In [5]:
people_raw = pd.read_csv(data_path + '/google_with_ids_titlecase.csv')

people_raw.head()

person_id           full_name      name_titlecase  \
0  P-12TR0ZjTkIPrJfmSJIFA_0000  bakhodir khalmetov  Bakhodir Khalmetov   
1  7ZslogBEXexnd9wu4AcrHQ_0000           hưng trần           Hưng Trần   
2  6HKrbsLp9BWotcSMBHd09A_0000          john burke          John Burke   
3  L440DVnM7He81jjiTwGAjQ_0000     stephane panier     Stephane Panier   
4  PmeTuyO1QSH3iZJkjXXOgA_0000         peter fotze         Peter Fotze   

                          linkedin_url                job_company_id  \
0                                  NaN  aKCIYBNF9ey6o5CjHCCO4goHYKlf   
1  linkedin.com/in/hưng-trần-465044166  aKCIYBNF9ey6o5CjHCCO4goHYKlf   
2                                  NaN                           NaN   
3      linkedin.com/in/stephane-panier  bPUqn45r2cWPYtHYoc03mQ7msrae   
4                                  NaN                           NaN   

  job_company_name  job_company_employee_count job_company_inferred_revenue  \
0           google                    200428.0                        $10B+   
1           google                    200428.0                        $10B+   
2              NaN                         NaN                          NaN   
3   new level work                       125.0                     $1M-$10M   
4              NaN                         NaN                          NaN   

   job_company_total_funding_raised  \
0                        26100000.0   
1                        26100000.0   
2                               NaN   
3                        16000000.0   
4                               NaN   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [6]:
organisations_raw = pd.read_csv(data_path + '/company_data.csv')

organisations_raw.head()

,id,name,display_name,employee_count,inferred_revenue,summary,total_funding_raised,funding_details
0,j0bfSucYe2pdOLfd8HXOSgNVpdDq,seedevice inc.,SeeDevice Inc.,7.0,$0-$1M,"a company developing cmos-swir image sensors and cameras using quantum based photodetector technology for computer & machine vision applications in automotive, industrial, and biomedical/biometric markets.",9.766270e+06,"[{'funding_type': 'series_unknown', 'investing_companies': [], 'investing_individuals': []}, {'funding_type': 'series_unknown', 'investing_companies': [], 'investing_individuals': []}, {'funding_type': 'seed', 'investing_companies': [], 'investing_individuals': []}]"
1,iX3LUUX1G29yDLHMVCllkgOYtAzh,open society foundations,Open Society Foundations,842.0,$100M-$250M,"the open society foundations work to build vibrant and inclusive democracies whose governments are accountable to their people. \n\nto achieve this mission, we give thousands of grants every year to groups and individuals in over 120 countries that work on the issues we focus on—promoting tolerance, transparency, and open debate. we also engage in strategic human rights litigation and impact investing, while incubating new ideas and engaging directly with governments and policymakers through advocacy to advance positive change. \n\nwe seek to shape public policies that assure greater fairness in political, legal, and economic systems and safeguard fundamental rights. we build alliances across borders and continents on issues such as corruption and freedom of information. we place a high priority on protecting and improving the lives of people in marginalized communities.\n\nthe open society foundations were founded by george soros, one of the world’s foremost philanthropists, who since 1984",NaN,[]
2,1TDgoIfjb6YxLiGkYiC8rQIdOtqV,crowdstrike,CrowdStrike,8495.0,$1B-$10B,"crowdstrike (nasdaq: crwd), a global cybersecurity leader, has redefined modern security with the world’s most advanced cloud-native platform for protecting critical areas of enterprise risk — endpoints and cloud workloads, identity and data.\n\npowered by the crowdstrike security cloud and world-class ai, the crowdstrike falcon® platform leverages real-time indicators of attack, threat intelligence, evolving adversary tradecraft and enriched telemetry from across the enterprise to deliver hyper-accurate detections, automated protection and remediation, elite threat hunting and prioritized observability of vulnerabilities.\n\npurpose-built in the cloud with a single lightweight-agent architecture, the falcon platform delivers rapid and scalable deployment, superior protection and performance, reduced complexity and immediate time-to-value.\n\ncrowdstrike: we stop breaches.",1.235600e+09,"[{'funding_type': 'series_b', 'investing_companies': [], 'investing_individuals': []}, {'funding_type': 'series_a', 'investing_companies': [], 'investing_individuals': []}, {'funding_type': 'series_d', 'investing_companies': ['F5WbPUYyP9PYwt31xCLQBQqgewTE'], 'investing_individuals': []}, {'funding_type': 'series_c', 'investing_companies': ['dVyB5JNFQJfq4HCkSCb4UgIY2pIT'], 'investing_individuals': []}, {'funding_type': 'post_ipo_debt', 'investing_companies': [], 'investing_individuals': []}, {'funding_type': 'series_d', 'investing_companies': [], 'investing_individuals': []}, {'funding_type': 'post_ipo_equity', 'investing_companies': ['REH11plNBABBxZt35CFBRwBbaPy2'], 'investing_individuals': []}, {'funding_type': 'series_e', 'investing_companies': [], 'investing_individuals': []}, {'funding_type': 'secondary_market', 'investing_companies': [], 'investing_individuals': []}]"
3,TzzwX4Pw2ogBcGCo1F5QVAYfptks,lloyds banking group,Lloyds Banking Group,35619.0,$1B-$10B,"our purpose is helping britain prosper. we do this by creating a more sustainable and inclusive future for people and businesses, shaping finance as a force for good.\n\nwe're part of an ever-changing industry and are currently on a journey to shape the financial services of

## Support Functions

In [7]:
# Helper function to safely get nested values
def get_nested_value(data, keys, default=None):
    for key in keys:
        if not isinstance(data, dict):  # Check if data is a dictionary
            return default
        data = data.get(key)
    return data if data is not None else default

# Function to process the 'experience' column and extract data
def process_experience_data(df):
    rows = []
    for _, row in df.iterrows():
        # Parse the JSON string in the 'experience' column
        experience_data = row['experience']
        print(experience_data)
        try:
            experiences = json.loads(experience_data
                                     .replace("'", "\"")
                                     .replace("None", "null")
                                     .replace("True", "true")
                                     .replace("False", "false")
                                    )
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON for row {row['person_id']}: {e}")
            experiences = []  # Handle invalid JSON gracefully
        
        # Extract details for each experience
        for exp in experiences:
            row_data = {
                "person_uuid": row['person_id'],
                "person_name": row['name_titlecase'],
                "linkedin_url": row['linkedin_url'],
                "org_uuid_target": row['job_company_id'],
                "org_name_target": row['job_company_name'],
                "total_funding_usd_target": row['job_company_total_funding_raised'],
                
                # Add nested fields dynamically
                "org_uuid_subsequent": get_nested_value(exp, ["company", "id"]),
                "org_name_subsequent": get_nested_value(exp, ["company", "name"]),
                "org_website_subsequent": get_nested_value(exp, ["company", "website"]),
                "org_location_subsequent_name": get_nested_value(exp, ["company", "location", "name"]),
                "org_country_code_subsequent": get_nested_value(exp, ["company", "location", "country"]),
                "org_city_subsequent": get_nested_value(exp, ["company", "location", "metro"]),
                "org_size_subsequent": get_nested_value(exp, ["company", "size"]),
                "founded_on_subsequent": get_nested_value(exp, ["company", "founded"]),
                "org_industry_subsequent": get_nested_value(exp, ["company", "industry"]),
                "company_type_subsequent": get_nested_value(exp, ["company", "type"]),
                "job_title_subsequent": get_nested_value(exp, ["title", "name"]),
                "job_role_subsequent": get_nested_value(exp, ["title", "role"]),
                "relation_type": get_nested_value(exp, ["title", "levels"]),
                "start_date": exp.get("start_date"),
                "end_date": exp.get("end_date"),
            }
            rows.append(row_data)
    return pd.DataFrame(rows)

In [8]:
flattened_data = process_experience_data(people_raw)
df = pd.DataFrame(flattened_data)

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [9]:
df

,person_uuid,person_name,linkedin_url,org_uuid_target,org_name_target,total_funding_usd_target,org_uuid_subsequent,org_name_subsequent,org_website_subsequent,org_location_subsequent_name,org_country_code_subsequent,org_city_subsequent,org_size_subsequent,founded_on_subsequent,org_industry_subsequent,company_type_subsequent,job_title_subsequent,job_role_subsequent,relation_type,start_date,end_date
0,P-12TR0ZjTkIPrJfmSJIFA_0000,Bakhodir Khalmetov,NaN,aKCIYBNF9ey6o5CjHCCO4goHYKlf,google,26100000.0,aKCIYBNF9ey6o5CjHCCO4goHYKlf,google,google.com,"mountain view, california, united states",united states,"san jose, california",10001+,1998.0,internet,private,general manager,None,[manager],1979-12,None
1,7ZslogBEXexnd9wu4AcrHQ_0000,Hưng Trần,linkedin.com/in/hưng-trần-465044166,aKCIYBNF9ey6o5CjHCCO4goHYKlf,google,26100000.0,aKCIYBNF9ey6o5CjHCCO4goHYKlf,google,google.com,"mountain view, california, united states",united states,"san jose, california",10001+,1998.0,internet,private,mobile application developer and manager,engineering,[manager],1988-03,None
2,7ZslogBEXexnd9wu4AcrHQ_0000,Hưng Trần,linkedin.com/in/hưng-trần-465044166,aKCIYBNF9ey6o5CjHCCO4goHYKlf,google,26100000.0,lIB1go5iwamGb0ndqVDpigOz8irA,my studio,jeffdemetriou.com,"georgia, united states",united states,None,1-10,NaN,fine art,private,chief executive officer and manager,None,[],2015-07,None
3,6HKrbsLp9BWotcSMBHd09A_0000,John Burke,NaN,NaN,NaN,NaN,shppxQIoBr8u28CIGPlp2gQgU4uP,aol,aol.com,"new york, new york, united states",united states,"new york, new york",1001-5000,1985.0,internet,private,senior vice president global sales strategy,sales,"[senior, vp]",2011-01-31,2011-12-31
4,6HKrbsLp9BWotcSMBHd09A_0000,John Burke,NaN,NaN,NaN,NaN,None,microwarehouse,None,None,None,None,None,NaN,None,None,senior account manager,sales,[senior],1994-12-31,1996-12-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2140,-Zt5ATeN9bszQo3s6ICTeA_0000,Karine Benhamou,linkedin.com/in/karine-benhamou-07347475,NaN,benhamou consulting ldt,NaN,uju6H8FGC5WmY9Um2WS0HgY4rFMi,youtube,youtube.com,"san bruno, california, united states",united states,"san francisco, california",1001-5000,2005.0,internet,private,project director,None,[director],2003-02,2010-03
2141,mopvJnGSZ1kbjUTe7L2lvQ_0000,Ajacea Dillard,linkedin.com/in/ajacea-dillard-b2420113a,aKCIYBNF9ey6o5CjHCCO4goHYKlf,google,26100000.0,aKCIYBNF9ey6o5CjHCCO4goHYKlf,google,google.com,"mountain view, california, united states",united states,"san jose, california",10001+,1998.0,internet,private,manager,None,[manager],1999-03,None
2142,UMBJUgBSGCFzYonblPoOFw_0000,Remco Aerts,NaN,NaN,NaN,NaN,None,ssg de rede,None,None,None,None,None,NaN,None,None,leraar,None,[],None,None
2143,UMBJUgBSGCFzYonblPoOFw_0000,Remco Aerts,NaN,NaN,NaN,NaN,aKCIYBNF9ey6o5CjHCCO4goHYKlf,google,google.com,"mountain view, california, united states",united states,"san jose, california",10001+,1998.0,internet,private,chief executive officer,None,[cxo],2000-03,2005-03


In [14]:
df[['person_uuid', 'linkedin_url', 'person_name', 'org_name_target', 'org_name_subsequent', 'founded_on_subsequent', 'job_title_subsequent', 'start_date', 'end_date']].head(50)

,person_uuid,linkedin_url,person_name,org_name_target,org_name_subsequent,founded_on_subsequent,job_title_subsequent,start_date,end_date
0,P-12TR0ZjTkIPrJfmSJIFA_0000,NaN,Bakhodir Khalmetov,google,google,1998.0,general manager,1979-12,None
1,7ZslogBEXexnd9wu4AcrHQ_0000,linkedin.com/in/hưng-trần-465044166,Hưng Trần,google,google,1998.0,mobile application developer and manager,1988-03,None
2,7ZslogBEXexnd9wu4AcrHQ_0000,linkedin.com/in/hưng-trần-465044166,Hưng Trần,google,my studio,NaN,chief executive officer and manager,2015-07,None
3,6HKrbsLp9BWotcSMBHd09A_0000,NaN,John Burke,NaN,aol,1985.0,senior vice president global sales strategy,2011-01-31,2011-12-31
4,6HKrbsLp9BWotcSMBHd09A_0000,NaN,John Burke,NaN,microwarehouse,NaN,senior account manager,1994-12-31,1996-12-31
5,6HKrbsLp9BWotcSMBHd09A_0000,NaN,John Burke,NaN,micro warehouse,NaN,senior account manager,1994-12-31,1996-12-31
6,6HKrbsLp9BWotcSMBHd09A_0000,NaN,John Burke,NaN,google,1998.0,"managing director, americas strategy",2009-06-30,2011-01-31
7,6HKrbsLp9BWotcSMBHd09A_0000,NaN,John Burke,NaN,google,1998.0,"managing director, industry development and marketing",2001-11-30,2009-06-30
8,6HKrbsLp9BWotcSMBHd09A_0000,NaN,John Burke,NaN,"zones, llc",1986.0,vice president of channel marketing,1998-12-31,2001-12-31
9,6HKrbsLp9BWotcSMBHd09A_0000,NaN,John Burke,NaN,motoring group,NaN,haymarket publishing,1992-11-30,1994-06-30


In [10]:
df.loc[df['person_uuid'] == 'tsaVKEBMg6MJr0xPwPlwSg_0000']

,person_uuid,person_name,linkedin_url,org_uuid_target,org_name_target,total_funding_usd_target,org_uuid_subsequent,org_name_subsequent,org_website_subsequent,org_location_subsequent_name,org_country_code_subsequent,org_city_subsequent,org_size_subsequent,founded_on_subsequent,org_industry_subsequent,company_type_subsequent,job_title_subsequent,job_role_subsequent,relation_type,start_date,end_date
23,tsaVKEBMg6MJr0xPwPlwSg_0000,Christopher Coleman,NaN,VyA410j5Ryn0nukJBnYCDAG67ePb,fidelity investments,152825000.0,smQC6yzWWbkRy1L2Ni7TbgqWZgLw,j.p. morgan,jpmorgan.com,"new york, new york, united states",united states,"new york, new york",10001+,1799.0,financial services,public,chief executive officer,None,[cxo],1999,None
24,tsaVKEBMg6MJr0xPwPlwSg_0000,Christopher Coleman,NaN,VyA410j5Ryn0nukJBnYCDAG67ePb,fidelity investments,152825000.0,aKCIYBNF9ey6o5CjHCCO4goHYKlf,google,google.com,"mountain view, california, united states",united states,"san jose, california",10001+,1998.0,internet,private,chief executive officer,None,[cxo],1997-10,None
25,tsaVKEBMg6MJr0xPwPlwSg_0000,Christopher Coleman,NaN,VyA410j5Ryn0nukJBnYCDAG67ePb,fidelity investments,152825000.0,8QPs7V62lOefbTnR9UpgmwCn3lDe,linkedin,linkedin.com,"sunnyvale, california, united states",united states,"san jose, california",10001+,2003.0,internet,public_subsidiary,chief executive officer,None,[cxo],1997-01,None
26,tsaVKEBMg6MJr0xPwPlwSg_0000,Christopher Coleman,NaN,VyA410j5Ryn0nukJBnYCDAG67ePb,fidelity investments,152825000.0,dapBox9LFjReLTHBE8OB5A2Y4XWL,"coca-cola bottling company united, inc.",cocacolaunited.com,"alabama, united states",united states,None,5001-10000,1902.0,consumer goods,private,chief executive officer,None,[cxo],1979-10,None
27,tsaVKEBMg6MJr0xPwPlwSg_0000,Christopher Coleman,NaN,VyA410j5Ryn0nukJBnYCDAG67ePb,fidelity investments,152825000.0,BTxhRCvBqFHEIlSGhrXLNgD4lJIn,thomson reuters,tr.com,"new york, new york, united states",united states,"new york, new york",10001+,2008.0,information technology and services,public,chief executive officer and chief financial officer,finance,[cxo],1979-10,None
28,tsaVKEBMg6MJr0xPwPlwSg_0000,Christopher Coleman,NaN,VyA410j5Ryn0nukJBnYCDAG67ePb,fidelity investments,152825000.0,mBdlnD9uVu0lvE1N9C48eQAn0KEZ,visa,visa.com,"foster city, california, united states",united states,"san francisco, california",10001+,1958.0,information technology and services,public,chief executive officer and chief financial officer,finance,[cxo],1979-10,None
29,tsaVKEBMg6MJr0xPwPlwSg_0000,Christopher Coleman,NaN,VyA410j5Ryn0nukJBnYCDAG67ePb,fidelity investments,152825000.0,YoI8B7XjFAy8ZjCcrh3XlAHIQZVl,coca-cola europacific partners,cocacolaep.com,"london, greater london, united kingdom",united kingdom,None,10001+,2016.0,consumer goods,public,chief executive officer,None,[cxo],1979-10,None
30,tsaVKEBMg6MJr0xPwPlwSg_0000,Christopher Coleman,NaN,VyA410j5Ryn0nukJBnYCDAG67ePb,fidelity investments,152825000.0,qKIGZFsZTPaFDMnSdvEMdQkkL1RH,coca-cola femsa,coca-colafemsa.com,"mexico, mexico",mexico,None,10001+,1993.0,food & beverages,public,chief executive officer,None,[cxo],1979-10,None
31,tsaVKEBMg6MJr0xPwPlwSg_0000,Christopher Coleman,NaN,VyA410j5Ryn0nukJBnYCDAG67ePb,fidelity investments,152825000.0,308AUASV3moZnOOimxrEQAapO6Ao,yahoo,yahoo.com,"sunnyvale, california, united states",united states,"san jose, california",10001+,1994.0,internet,public_subsidiary,chief executive officer,None,[cxo],1997,None
32,tsaVKEBMg6MJr0xPwPlwSg_0000,Christopher Coleman,NaN,VyA410j5Ryn0nukJBnYCDAG67ePb,fidelity investments,152825000.0,q9dK2WXoX3CC6B9nfAtljgGTIX32,apple,apple.com,"cupertino, california, united states",united states,"san jose, california",10001+,1976.0,consumer electronics,public,chief executive officer and chief financial officer,finance,[cxo],1979-10,None


In [11]:
subsequent_orgs_count = (
    df
        .groupby('person_uuid', as_index = False)
        .agg(
            {
                'person_name': 'count'
            },
            skipna = True
        )
).sort_values(by = 'person_name', ascending = False)

#subsequent_orgs_count = pd.merge(subsequent_orgs_count, df, how = 'left', left_on = 'person_uuid', right_on = 'person_uuid')

subsequent_orgs_count.head(50)

,person_uuid,person_name
92,6jhbXCbOxOLPYBiiN55xDQ_0000,27
536,nnA7AX5eWcuWobp4ANSrrg_0000,26
127,9ai9w4O4iIymlrwtHrBGPA_0000,23
254,KBHMXAwh7OjFH8J0YbNP1A_0000,23
594,tsaVKEBMg6MJr0xPwPlwSg_0000,19
617,vnsR3cMND-Ked56IFsU0tQ_0000,19
319,QYy-AcNSnHFBr12cB40M4A_0000,18
207,FwX9i7yTKIP1XBihgBAN8A_0000,18
101,7bCXx3IOiAOwtZN0oG7Rtg_0000,17
50,3AHgm-KWL4is71A4jkOxjQ_0000,17
